# Scraping free patterns from Herrschners

### Setting the path for the image data and importing packages 

In [28]:
import os
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

headers = {"User-Agent": "Mozilla/5.0"}

image_dir = "../data/images"
os.makedirs(image_dir, exist_ok=True)

### Scraping every product link

In [ ]:
def scrape_product(product_link, image_dir="../data/images"):
    row = {}
    os.makedirs(image_dir, exist_ok=True)
    
    try:
        r = requests.get(product_link, headers=headers, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
    except Exception as e:
        print(f"Connection error for {product_link}: {e}")
        return row

    # Information
    title_element = soup.find("h1", class_="productView-title")
    if title_element:
        raw_title = title_element.get_text(strip=True)
        clean_title = re.sub(r'\(?Free Download\)?', '', raw_title, flags=re.IGNORECASE).strip()
        row["title"] = clean_title
    else:
        row["title"] = ""
    names = soup.find_all(class_="productView-info-name")
    values = soup.find_all(class_="productView-info-value")
    for name, value in zip(names, values):
        key = name.get_text(strip=True).lower().replace(":", "").replace(" ", "_")
        row[key] = value.get_text(strip=True)
    
    row["merged_info"] = " ".join(v.get_text(strip=True) for v in values)

    # Description
    desc_container = soup.find(class_="productView-desc-content")
    if desc_container:
        row["description"] = desc_container.get_text(" ", strip=True)

    # Review Count
    review_link = soup.find(class_="productView-reviewLink")
    if review_link:
        match = re.search(r"\d+", review_link.get_text())
        row["review_count"] = int(match.group()) if match else 0
    else:
        row["review_count"] = 0

    # Star Rating
    rating_div = soup.find(class_="productView-rating")
    if rating_div:
        stars = rating_div.find_all(class_=re.compile(r"icon--ratingFull"))
        row["star_rating"] = len(stars)
    else:
        row["star_rating"] = 0

    # Image Extraction and Download
    img_container = soup.find("span", class_="productView-imageCarousel-main-item-img-container")
    img_tag = img_container.find("img") if img_container else None
    
    img_url = None
    if img_tag:
        img_url = img_tag.get("src")
    
    row['image_url'] = img_url

    if img_url:
        safe_filename = re.sub(r'[^\w\s-]', '', row["title"]).strip().replace(' ', '_')
        save_path = os.path.join(image_dir, f"{safe_filename}.jpg")

        try:
            img_res = requests.get(img_url, headers=headers, stream=True, timeout=10)
            if img_res.status_code == 200 and 'image' in img_res.headers.get('Content-Type', ''):
                with open(save_path, 'wb') as f:
                    for chunk in img_res.iter_content(1024):
                        f.write(chunk)
                row["local_path"] = save_path
            else:
                print(f"Invalid image response for: {row['title']}")
        except Exception as e:
            print(f"Error downloading image for {row['title']}: {e}")

    return row

In [ ]:
base_url = "https://herrschners.com/herrschners/knit-crochet/books-patterns/free-pattern-downloads/"
rows = []
page = 1

while True:
    print(f"Scraping page {page}...")
    url = base_url if page == 1 else f"{base_url}?page={page}"

    #if page == 2:
    #    break

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    cards = soup.select("article.card")
    if not cards:
        print("No more products found. Ending scrape.")
        break

    for card in cards:
        product_link = card.get("data-href")
        row = {"product_link": product_link}

        if product_link:
            features = scrape_product(product_link)
            row.update(features)

        rows.append(row)
        time.sleep(0.5)

    page += 1

df = pd.DataFrame(rows)

df.columns = [col.replace(':', '') for col in df.columns]
print("Scraping complete!")

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...
Scraping page 9...
Scraping page 10...
Scraping page 11...
Scraping page 12...
Scraping page 13...
No more products found. Ending scrape.
Scraping complete!


In [31]:
df.head()

,product_link,title,sku,upc,project_type,skill,difficulty,merged_info,description,review_count,star_rating,image_url,local_path,occasion,season,theme,fiber_content,yarn_weight,thread_size
0,https://herrschners.com/herrschners-sunrise-st...,Herrschners Sunrise Stroll Tote,F01730,,Tote,Crochet,Easy,F01730 Tote Crochet Easy,Accessorize in style with this textured two-to...,0,0,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ima...,../data/images/Herrschners_Sunrise_Stroll_Tote...,NaN,NaN,NaN,NaN,NaN,NaN
1,https://herrschners.com/village-yarn-blue-burs...,Village Yarn Blue Burst Pot Holders,F01729,,Pot Holders,Crochet,Easy,F01729 Pot Holders Crochet Easy,Brighten your kitchen with a splash of color! ...,0,0,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ima...,../data/images/Village_Yarn_Blue_Burst_Pot_Hol...,NaN,NaN,NaN,NaN,NaN,NaN
2,https://herrschners.com/willow-yarns-garden-so...,Willow Yarns Garden Song Shawlette,F01728,,Shawl,Knit,Easy,F01728 Shawl Knit Easy,Be ready for backyard garden parties with this...,0,0,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ima...,../data/images/Willow_Yarns_Garden_Song_Shawle...,NaN,NaN,NaN,NaN,NaN,NaN
3,https://herrschners.com/herrschners-english-ga...,Herrschners English Garden Doily,F01727,,Doily,Crochet,Easy,F01727 Doily Crochet Easy,No green thumb is needed to grow a garden of t...,0,0,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ima...,../data/images/Herrschners_English_Garden_Doil...,NaN,NaN,NaN,NaN,NaN,NaN
4,https://herrschners.com/herrschners-shamrock-g...,Herrschners Shamrock Glow Blanket,F01726,,Blanket,Crochet,Easy,F01726 St. Patrick's Day Blanket Crochet Easy,It’s your lucky day! This free easy crochet bl...,0,0,https://cdn11.bigcommerce.com/s-wgzqwlngdf/ima...,../data/images/Herrschners_Shamrock_Glow_Blank...,St. Patrick's Day,NaN,NaN,NaN,NaN,NaN


In [33]:
df['full_description'] = df['title'] + " " + df['description'].fillna('') + " " + df['merged_info'].fillna('')

In [34]:
df.to_csv("../data/herrschners_crochet_patterns.csv", index=False)